In [7]:
import os
from collections import defaultdict
from datetime import datetime
from pathlib import Path

import pandas as pd
from IPython.display import HTML, display
from tqdm.auto import tqdm

# ── Config ─────────────────────────────────────────────────────────────────
LOCAL_DATA_ROOT = "/home/taiaburrahman/dataset_manager_pro/data/processed/v9"
SOURCE_CSV_PATH = "/home/taiaburrahman/dataset_manager_pro/data/projects/GAID/07.csv/GAID_Dataset_v9_full_Train.csv"
SAVE_REPORT     = "/home/taiaburrahman/dataset_manager_pro/notebook/reports/datasets/v9/compare"

# Number of folder levels to use (counted upward from the file name).
#   FOLDER_DEPTH = 1 -> <folder>/filename.jpg
#   FOLDER_DEPTH = 2 -> <folder>/<folder>/filename.jpg
#   FOLDER_DEPTH = 3 -> <folder>/<folder>/<folder>/filename.jpg
FOLDER_DEPTH    = 2

IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff"}
# ───────────────────────────────────────────────────────────────────────────


def folder_key(path_str: str, depth: int) -> str:
    """Return the last `depth` folder names above the file (e.g. 'fake/genAI')."""
    parts = Path(str(path_str)).parent.parts
    if not parts or depth <= 0:
        return ""
    return "/".join(parts[-depth:]) if len(parts) >= depth else "/".join(parts)


# ─────────────────────────────────────────────────────────────────────────
# 1) Build CSV index: folder_key -> set of file basenames + an example path
# ─────────────────────────────────────────────────────────────────────────
df = pd.read_csv(SOURCE_CSV_PATH)
if "image_path" not in df.columns:
    raise ValueError("CSV must contain an 'image_path' column")

print(f"Loaded CSV rows: {len(df):,}")
print(f"Folder depth   : {FOLDER_DEPTH}")

csv_folder_examples: dict[str, str] = {}
csv_files_by_folder: dict[str, set[str]] = defaultdict(set)
for img_path in tqdm(df["image_path"].astype(str), desc="Scan CSV", unit="row"):
    p = Path(img_path)
    key = folder_key(img_path, FOLDER_DEPTH)
    if not key:
        continue
    csv_files_by_folder[key].add(p.name)
    if key not in csv_folder_examples:
        csv_folder_examples[key] = str(p.parent)

csv_folders = set(csv_folder_examples.keys())
print(f"Unique CSV folder keys: {len(csv_folders):,}")


# ─────────────────────────────────────────────────────────────────────────
# 2) Build LOCAL index: folder_key -> set of file basenames + an example path
# ─────────────────────────────────────────────────────────────────────────
local_root = Path(LOCAL_DATA_ROOT)
if not local_root.is_dir():
    raise FileNotFoundError(f"LOCAL_DATA_ROOT not found: {local_root}")

local_folder_examples: dict[str, str] = {}
local_files_by_folder: dict[str, set[str]] = defaultdict(set)
file_iter = (p for p in local_root.rglob("*") if p.is_file() and p.suffix.lower() in IMG_EXTS)
for p in tqdm(file_iter, desc=f"Scan {local_root.name}", unit="file"):
    key = folder_key(str(p), FOLDER_DEPTH)
    if not key:
        continue
    local_files_by_folder[key].add(p.name)
    if key not in local_folder_examples:
        local_folder_examples[key] = str(p.parent)

local_folders = set(local_folder_examples.keys())
print(f"Unique local folder keys: {len(local_folders):,}")


# ─────────────────────────────────────────────────────────────────────────
# 3) Folder-level diff + per-folder image match counts
# ─────────────────────────────────────────────────────────────────────────
missing_in_local = sorted(csv_folders - local_folders)   # referenced by CSV but missing locally
extra_in_local   = sorted(local_folders - csv_folders)   # present locally but not referenced by CSV
common_folders   = sorted(csv_folders & local_folders)

print(f"\n  ✓ Common : {len(common_folders):,}")
print(f"  ✗ Missing (in CSV, not local): {len(missing_in_local):,}")
print(f"  + Extra   (in local, not CSV): {len(extra_in_local):,}")


def _counts(key: str) -> tuple[int, int, int, int, int]:
    """Return (csv_total, local_total, match, only_csv, only_local) for a folder key."""
    cset = csv_files_by_folder.get(key, set())
    lset = local_files_by_folder.get(key, set())
    match = len(cset & lset)
    only_csv = len(cset - lset)
    only_local = len(lset - cset)
    return len(cset), len(lset), match, only_csv, only_local


tot_csv = tot_local = tot_match = tot_only_csv = tot_only_local = 0
for key in csv_folders | local_folders:
    c_n, l_n, m, oc, ol = _counts(key)
    tot_csv += c_n; tot_local += l_n
    tot_match += m; tot_only_csv += oc; tot_only_local += ol

print(f"\n  CSV images          : {tot_csv:,}")
print(f"  Local images        : {tot_local:,}")
print(f"  Matched images      : {tot_match:,}")
print(f"  Only-in-CSV images  : {tot_only_csv:,}")
print(f"  Only-in-local images: {tot_only_local:,}")


# ─────────────────────────────────────────────────────────────────────────
# 4) Build markdown report and save
# ─────────────────────────────────────────────────────────────────────────
save_dir = Path(SAVE_REPORT)
save_dir.mkdir(parents=True, exist_ok=True)

csv_stem = Path(SOURCE_CSV_PATH).stem
ts       = datetime.now().strftime("%Y%m%d_%H%M%S")
out_md   = save_dir / f"compare_{csv_stem}_depth{FOLDER_DEPTH}_{ts}.md"

# (folder_key, csv_example, local_example, csv_total, local_total, match, only_csv, only_local)
Row = tuple[str, str, str, int, int, int, int, int]


def _row(key: str) -> Row:
    c_n, l_n, m, oc, ol = _counts(key)
    return (
        key,
        csv_folder_examples.get(key, "—"),
        local_folder_examples.get(key, "—"),
        c_n, l_n, m, oc, ol,
    )


missing_rows: list[Row] = [_row(k) for k in missing_in_local]
extra_rows:   list[Row] = [_row(k) for k in extra_in_local]
common_rows:  list[Row] = [_row(k) for k in common_folders]


def _md_table(rows: list[Row]) -> str:
    if not rows:
        return "_(none)_\n"
    lines = [
        "| # | Folder Key | CSV Path (example) | Local Path (example) "
        "| CSV imgs | Local imgs | Match | Only CSV | Only Local |",
        "|---:|---|---|---|---:|---:|---:|---:|---:|",
    ]
    for i, (k, a, b, c_n, l_n, m, oc, ol) in enumerate(rows, 1):
        k = (k or "").replace("|", r"\|")
        a = (a or "").replace("|", r"\|")
        b = (b or "").replace("|", r"\|")
        lines.append(
            f"| {i} | `{k}` | `{a}` | `{b}` "
            f"| {c_n:,} | {l_n:,} | {m:,} | {oc:,} | {ol:,} |"
        )
    return "\n".join(lines) + "\n"


md: list[str] = []
md.append(f"# Folder Comparison Report — depth = {FOLDER_DEPTH}")
md.append("")
md.append(f"- **Generated**     : `{ts}`")
md.append(f"- **CSV**           : `{SOURCE_CSV_PATH}`")
md.append(f"- **Local root**    : `{LOCAL_DATA_ROOT}`")
md.append(f"- **Folder depth**  : `{FOLDER_DEPTH}` (folders counted upward from file name)")
md.append("")
md.append("> File-level match is computed by file basename within each folder key. ")
md.append("> `Match` = files present in both, `Only CSV` = referenced by CSV but missing on disk, ")
md.append("> `Only Local` = on disk but not referenced by CSV.")
md.append("")
md.append("## Summary")
md.append("")
md.append("| Metric | Count |")
md.append("|---|---:|")
md.append(f"| Unique folder keys in CSV   | {len(csv_folders):,} |")
md.append(f"| Unique folder keys in local | {len(local_folders):,} |")
md.append(f"| Common (in both)            | {len(common_folders):,} |")
md.append(f"| Missing folders (in CSV, not local) | {len(missing_in_local):,} |")
md.append(f"| Extra folders   (in local, not CSV) | {len(extra_in_local):,} |")
md.append(f"| CSV images (total)          | {tot_csv:,} |")
md.append(f"| Local images (total)        | {tot_local:,} |")
md.append(f"| **Matched images**          | **{tot_match:,}** |")
md.append(f"| Only-in-CSV images          | {tot_only_csv:,} |")
md.append(f"| Only-in-local images        | {tot_only_local:,} |")
md.append("")
md.append(f"## Missing folders — referenced by CSV but not found locally ({len(missing_in_local):,})")
md.append("")
md.append(_md_table(missing_rows))
md.append("")
md.append(f"## Extra folders — present locally but not referenced by CSV ({len(extra_in_local):,})")
md.append("")
md.append(_md_table(extra_rows))
md.append("")
md.append(f"## Common folders ({len(common_folders):,})")
md.append("")
md.append(_md_table(common_rows))
md.append("")

out_md.write_text("\n".join(md), encoding="utf-8")
print(f"\nReport saved → {out_md}")


# ─────────────────────────────────────────────────────────────────────────
# 5) Inline preview in the notebook
# ─────────────────────────────────────────────────────────────────────────
def _section_html(title: str, rows: list[Row], color: str) -> str:
    body = ""
    for k, a, b, c_n, l_n, m, oc, ol in rows[:200]:
        body += (
            f"<tr>"
            f"<td style='padding:4px 8px;font-family:monospace'>{k}</td>"
            f"<td style='padding:4px 8px;font-family:monospace;color:#555;font-size:11px'>{a}</td>"
            f"<td style='padding:4px 8px;font-family:monospace;color:#555;font-size:11px'>{b}</td>"
            f"<td style='padding:4px 8px;text-align:right'>{c_n:,}</td>"
            f"<td style='padding:4px 8px;text-align:right'>{l_n:,}</td>"
            f"<td style='padding:4px 8px;text-align:right;color:#27ae60;font-weight:bold'>{m:,}</td>"
            f"<td style='padding:4px 8px;text-align:right;color:#e74c3c'>{oc:,}</td>"
            f"<td style='padding:4px 8px;text-align:right;color:#f39c12'>{ol:,}</td>"
            f"</tr>"
        )
    extra_note = (
        f"<div style='font-size:11px;color:#888;margin:4px 0'>"
        f"showing first 200 of {len(rows):,}</div>"
    ) if len(rows) > 200 else ""
    return (
        f"<h4 style='margin:14px 0 4px 0;color:{color}'>{title} ({len(rows):,})</h4>"
        f"{extra_note}"
        f"<table style='border-collapse:collapse;font-size:12px'>"
        f"<thead><tr style='background:{color};color:#fff;text-align:left'>"
        f"<th style='padding:4px 8px'>Folder Key</th>"
        f"<th style='padding:4px 8px'>CSV Path (example)</th>"
        f"<th style='padding:4px 8px'>Local Path (example)</th>"
        f"<th style='padding:4px 8px;text-align:right'>CSV imgs</th>"
        f"<th style='padding:4px 8px;text-align:right'>Local imgs</th>"
        f"<th style='padding:4px 8px;text-align:right'>Match</th>"
        f"<th style='padding:4px 8px;text-align:right'>Only CSV</th>"
        f"<th style='padding:4px 8px;text-align:right'>Only Local</th>"
        f"</tr></thead><tbody>{body}</tbody></table>"
    )


display(HTML(
    f"<h3>Folder Comparison — depth = {FOLDER_DEPTH}</h3>"
    f"<div style='font-size:12px;color:#444'>"
    f"CSV: <code>{SOURCE_CSV_PATH}</code><br/>"
    f"Local: <code>{LOCAL_DATA_ROOT}</code><br/>"
    f"Report: <code>{out_md}</code><br/>"
    f"Images — CSV: <b>{tot_csv:,}</b>, Local: <b>{tot_local:,}</b>, "
    f"Match: <b style='color:#27ae60'>{tot_match:,}</b>, "
    f"Only CSV: <b style='color:#e74c3c'>{tot_only_csv:,}</b>, "
    f"Only Local: <b style='color:#f39c12'>{tot_only_local:,}</b>"
    f"</div>"
    + _section_html("Missing — in CSV, not local", missing_rows, "#e74c3c")
    + _section_html("Extra — in local, not CSV", extra_rows, "#f39c12")
    + _section_html("Common — in both", common_rows, "#2c3e50")
))


Loaded CSV rows: 704,779
Folder depth   : 2


Scan CSV:   0%|          | 0/704779 [00:00<?, ?row/s]

Unique CSV folder keys: 110


Scan v9: 0file [00:00, ?file/s]

Unique local folder keys: 108

  ✓ Common : 88
  ✗ Missing (in CSV, not local): 22
  + Extra   (in local, not CSV): 20

  CSV images          : 704,779
  Local images        : 695,276
  Matched images      : 435,845
  Only-in-CSV images  : 268,934
  Only-in-local images: 259,431

Report saved → /home/taiaburrahman/dataset_manager_pro/notebook/reports/datasets/v9/compare/compare_GAID_Dataset_v9_full_Train_depth2_20260429_133611.md


Folder Key,CSV Path (example),Local Path (example),CSV imgs,Local imgs,Match,Only CSV,Only Local
Gen_samples_Gemini_100325/code_gen_100325,/home/ubuntu/vision/data/NewDB/NewDB_Fake/Gen_samples/fake/Gen_samples_Gemini_100325/code_gen_100325,—,"3,914",0,0,"3,914",0
Gen_samples_Gemini_100325/humangen_nature_070325,/home/ubuntu/vision/data/NewDB/NewDB_Fake/Gen_samples/fake/Gen_samples_Gemini_100325/humangen_nature_070325,—,402,0,0,402,0
gen_ai_detector_dataset/human,/home/ubuntu/vision/data/gen_ai_detector_dataset/human,—,"221,275",0,0,"221,275",0
gen_ai_detector_dataset/human_2,/home/ubuntu/vision/data/gen_ai_detector_dataset/human_2,—,"14,730",0,0,"14,730",0
gen_ai_detector_dataset/human_3,/home/ubuntu/vision/data/gen_ai_detector_dataset/human_3,—,"1,135",0,0,"1,135",0
real/low_res,/home/ubuntu/vision/data/NewDB/NewDB_Real/SupRes_aditya/real/low_res,—,831,0,0,831,0
real/real_faces_Japanese-Actors,/home/ubuntu/vision/data/NewDB/NewDB_Real/real_faces_multinational_130325/real/real_faces_Japanese-Actors,—,182,0,0,182,0
real/real_faces_JapaneseCelebrities_,/home/ubuntu/vision/data/NewDB/NewDB_Real/real_faces_multinational_130325/real/real_faces_JapaneseCelebrities_,—,"1,171",0,0,"1,171",0
real/real_faces_africanews.en_images,/home/ubuntu/vision/data/NewDB/NewDB_Real/real_faces_multinational_130325/real/real_faces_africanews.en_images,—,199,0,0,199,0
real/real_faces_american_1996communitty,/home/ubuntu/vision/data/NewDB/NewDB_Real/real_faces_multinational_130325/real/real_faces_american_1996communitty,—,774,0,0,774,0
